# 08 Non-Hermitian Black-Scholes Hamiltonian

Build the diagonal momentum-space Black-Scholes Hamiltonian and validate constant-volatility commutation.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
n_qubits = 5
_, _, dx = make_log_price_grid(n_qubits, center_price=24000, x_width=config["quantum"]["x_width"])
p = momentum_eigenvalues(2 ** n_qubits, dx)
sigma = 0.18; r = 0.065
H = black_scholes_hamiltonian_diagonal(p, sigma, r)
Hh, Ha = hamiltonian_parts_diagonal(p, sigma, r)
assert H.shape == Hh.shape == Ha.shape == (2 ** n_qubits,)
assert commutator_norm_for_diagonals(Hh, Ha) < 1e-12
print("VALIDATION PASSED: Hamiltonian dimensions and Hermitian/anti-Hermitian commutation")
hamiltonian_df = pd.DataFrame({"p": p, "H_real": H.real, "H_imag": H.imag, "Hermitian_diag": Hh, "AntiHermitian_imag_diag": Ha.imag})
save_table(hamiltonian_df, "08_hamiltonian_diagonal.csv")
plt.figure()
plt.plot(p, Hh, marker="o", label="Hermitian diagonal")
plt.plot(p, Ha.imag, marker="x", label="Anti-Hermitian imaginary diagonal")
plt.title("Momentum-space Black-Scholes Hamiltonian components")
plt.xlabel("Momentum eigenvalue")
plt.legend()
save_current_figure("08_hamiltonian_components.png")
hamiltonian_df.head()
